In [0]:
%pip install pypdf

In [0]:
dbutils.library.restartPython()

In [0]:
import re
from pypdf import PdfReader

VOLUME_PATH = "/Volumes/career_copilot/raw/career_corpus/project_docs"

def infer_project(filename: str) -> str:
    match = re.search(r"week\s*([1-9])", filename.lower())
    return f"week{match.group(1)}" if match else "general"

def infer_doc_type(filename: str) -> str:
    f = filename.lower()
    if "resume" in f:
        return "resume"
    if "readme" in f:
        return "readme"
    if "design" in f and "decision" in f:
        return "design_decisions"
    if "log" in f:
        return "daily_log"
    if re.match(r"week\d+-", f):
        return "case_study"
    return "other"

def read_file_text(path: str) -> str:
    if path.lower().endswith(".pdf"):
        reader = PdfReader(path)
        return "\n".join(page.extract_text() or "" for page in reader.pages)
    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        return f.read()

In [0]:
def chunk_text(text:str, chunk_size: int = 500, overlap: int = 50):
    paragraphs = re.split(r"\n\s*\n", text)
    chunks, current, current_len = [], [], 0

    for para in paragraphs:
        words = para.split()
        if current_len + len(words) > chunk_size and current:
            chunks.append(" ".join(current))
            current = current[-overlap:] if overlap < len(current) else current
            current_len = len(current)
        current.extend(words)
        current_len += len(words)
    if current:
        chunks.append(" ".join(current))
    return [c for c in chunks if c.strip()]


In [0]:
rows = []
files = dbutils.fs.ls(VOLUME_PATH)

for f in files:
    if f.name.endswith("/"):  # skip subdirectories
        continue
    filename = f.name
    full_path = f.path.replace("dbfs:", "")
    text = read_file_text(full_path)
    chunks = chunk_text(text)
    project = infer_project(filename)
    doc_type = infer_doc_type(filename)

    for i, chunk in enumerate(chunks):
        rows.append({
            "source_file": filename,
            "project_name": project,
            "doc_type": doc_type,
            "chunk_index": i,
            "chunk_text": chunk,
        })

print(f"Processed {len(files)} files into {len(rows)} chunks")

In [0]:
df = spark.createDataFrame(rows)
spark.sql("CREATE SCHEMA IF NOT EXISTS career_copilot.processed")
df.write.mode("overwrite").saveAsTable("career_copilot.processed.doc_chunks")
display(
    spark.sql("""
              SELECT source_file, project_name, doc_type, COUNT(*) AS num_chunks
              FROM career_copilot.processed.doc_chunks
              GROUP BY source_file, project_name, doc_type
              ORDER BY source_file
              """)
)